In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 데이터 로드
df_x = pd.read_csv('dataset/combined_mean_results.csv')
df_y = pd.read_csv('dataset/Avg_mood.csv')

X = df_x.iloc[:, :10]
Y = df_y.iloc[:, -2:]

# ✅ 1단계: NaN 확인 및 처리
print("X NaN 개수:\n", X.isnull().sum())
print("Y NaN 개수:\n", Y.isnull().sum())

X = X.dropna()  # NaN → 결측치가 있는 행 삭제
Y = Y.dropna()

X NaN 개수:
 FileName                            0
F0final_sma_stddev                  0
F0final_sma_amean                   0
voicingFinalUnclipped_sma_stddev    0
voicingFinalUnclipped_sma_amean     0
jitterLocal_sma_stddev              0
jitterLocal_sma_amean               0
jitterDDP_sma_stddev                0
jitterDDP_sma_amean                 0
shimmerLocal_sma_stddev             0
dtype: int64
Y NaN 개수:
 valence_mean    0
arousal_mean    0
dtype: int64


In [ ]:
# ✅ 2단계: 데이터 정규화
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_x.fit_transform(X)
Y_scaled = scaler_y.fit_transform(Y)

In [8]:
# train/test 분리
train_X, test_X, train_Y, test_Y = train_test_split(X, Y, test_size=0.3, random_state=42)

# tensor Dataset 구성 (train 데이터만 사용)
dataset = tf.data.Dataset.from_tensor_slices((train_X.values, train_Y.values))
dataset = dataset.shuffle(buffer_size=len(train_X)).batch(32)

In [16]:
# ✅ 3단계: 모델 구성 (BatchNormalization 추가)
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, input_dim=10, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(2, activation='linear'),
])


c:\Users\jh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(
    loss='mean_squared_error',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['mae']
)


In [21]:
# ✅ 5단계: NaN 발생 시 학습 중단하는 콜백 추가
history = model.fit(
    dataset,
    epochs=200,
    validation_data=(test_X, test_Y),
    verbose=1
)

Epoch 1/200
 1/40 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.5458 - mae: 1.0092

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.1016 - mae: 0.8467 - val_loss: 1.2686 - val_mae: 0.8999
Epoch 2/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.0920 - mae: 0.8410 - val_loss: 1.2426 - val_mae: 0.8986
Epoch 3/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.1217 - mae: 0.8482 - val_loss: 1.1514 - val_mae: 0.8624
Epoch 4/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0697 - mae: 0.8326 - val_loss: 1.2254 - val_mae: 0.8868
Epoch 5/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.1107 - mae: 0.8449 - val_loss: 1.4807 - val_mae: 0.9869
Epoch 6/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.1117 - mae: 0.8488 - val_loss: 1.2622 - val_mae: 0.9020
Epoch 7/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1054 - mae: 0.8453 - val_loss: 1.1875 - val_mae: 0.8788
Epoch 8/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1029 - mae: 0.8489 - val_loss: 1.1582 - val_mae: 0.8684
Epoch 9/200
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.1222 - mae: 0.8

In [12]:
# 평가
loss, mae = model.evaluate(test_X, test_Y, verbose=1)
print(f"\n테스트 Loss(MSE): {loss:.4f}")
print(f"테스트 MAE: {mae:.4f}")

# 예측 후 역변환 (원래 스케일로 복원)
predictions_scaled = model.predict(test_X)
predictions = scaler_y.inverse_transform(predictions_scaled)
test_Y_orig = scaler_y.inverse_transform(test_Y)

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1486 - mae: 0.8614 

테스트 Loss(MSE): 1.1486
테스트 MAE: 0.8614
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [15]:
# 결과 출력
print("\n--- 예측 결과 샘플 ---")
for i in range(10):
    real = test_Y_orig[i].round(2)
    pred = predictions[i].round(2)
    print(f"{i}번째 | 실제값: {real} | 예측값: {pred}")


--- 예측 결과 샘플 ---
0번째 | 실제값: [12.18  9.43] | 예측값: [10.35 10.31]
1번째 | 실제값: [11.36 12.38] | 예측값: [ 9.64 11.11]
2번째 | 실제값: [12.65 13.27] | 예측값: [11.43 11.73]
3번째 | 실제값: [ 8.07 10.84] | 예측값: [ 8.58 10.21]
4번째 | 실제값: [12.18 10.2 ] | 예측값: [11.08 11.9 ]
5번째 | 실제값: [13.36 12.51] | 예측값: [ 9.92 11.24]
6번째 | 실제값: [8.54 9.43] | 예측값: [ 9.5  10.99]
7번째 | 실제값: [8.54 9.3 ] | 예측값: [ 9.98 11.13]
8번째 | 실제값: [10.3   8.66] | 예측값: [10.93 10.35]
9번째 | 실제값: [11.71 11.35] | 예측값: [10.65 11.05]
